# 0. Imports

## 0.1 Packages

In [181]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

## 0.2 Data

In [182]:
ObsList = pd.read_csv(r"../Data Raw/ObsList.csv", sep=";")

# 1. Code Preparation

In [183]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def fetch_species(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('rank') == 'SPECIES':
                if data.get('status') != 'ACCEPTED':
                    accepted_name = data.get('species') or data.get('canonicalName')
                else:
                    accepted_name = data.get('canonicalName')
                
                return accepted_name if accepted_name else None
            else:
                return None
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def check_species_async(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_species(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def check_species(species_list):
    return asyncio.get_event_loop().run_until_complete(check_species_async(species_list))

In [184]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def fetch_family(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('family'):
                return data.get('family')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def check_family_async(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_family(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def check_family(species_list):
    return asyncio.get_event_loop().run_until_complete(check_family_async(species_list))

In [185]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def fetch_genus(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('genus'):
                return data.get('genus')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def check_genus_async(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_genus(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def check_genus(species_list):
    return asyncio.get_event_loop().run_until_complete(check_genus_async(species_list))

In [186]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def fetch_lepidoptera(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('order') == 'Lepidoptera':
                return 1
            elif data.get('order') is None:
                return None
            else:
                return 0
            
    except Exception as e:
        print(f"Error fetching data for {species}: {e}")
        return False

async def check_lepidoptera_async(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_lepidoptera(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def check_lepidoptera(species_list):
    return asyncio.get_event_loop().run_until_complete(check_lepidoptera_async(species_list))

# 2. Taxonomy Extract

In [187]:
Taxonomy = pd.DataFrame(ObsList['Species'].unique(), columns=['Species'])

In [188]:
Taxonomy['Species'] = Taxonomy['Species'].apply(lambda x: ' '.join(x.split()[:2]))

## 2.1 Accepted Species

In [189]:
Taxonomy['AcceptedSpecies'] = check_species(Taxonomy['Species'])

In [190]:
Taxonomy[Taxonomy['AcceptedSpecies'].isna()]

,Species,AcceptedSpecies
326,Heliocheilus cystiphora,None
355,Rheumaptera affirmata,None
362,Spodoptera sunia,None
363,Heliocontia margana,None
402,Trissodoris guamensis,None
585,Semiothisa santaremaria,None
628,Limenitis camilla,None
645,Phalaenophana fadusalis,None
661,Eumeta japonica,None
821,Agonopterix umbellana,None


In [191]:
Taxonomy['AcceptedSpecies'] = Taxonomy['AcceptedSpecies'].fillna(Taxonomy['Species'])
Taxonomy['AcceptedSpecies'] = Taxonomy['AcceptedSpecies'].apply(lambda x: ' '.join(x.split()[:2]))

Taxonomy.reset_index(drop=True, inplace=True)

## 2.2 Family

In [192]:
Taxonomy['Family'] = check_family(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

## 2.3 Genus

In [193]:
Taxonomy['Genus'] = check_genus(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

## 2.4 Lepidoptera Cross-Check

In [194]:
Taxonomy['Lepidoptera'] = check_lepidoptera(Taxonomy['AcceptedSpecies'])
Taxonomy.reset_index(drop=True, inplace=True)

In [195]:
Taxonomy[Taxonomy['Lepidoptera'] == 0]

,Species,AcceptedSpecies,Family,Genus,Lepidoptera


In [196]:
Taxonomy.drop(columns=['Lepidoptera'], inplace=True)

In [197]:
Taxonomy.drop_duplicates(inplace=True)

## 2.5 Export

In [198]:
Taxonomy.to_csv(r'../Data Raw/TaxonomyRaw.csv', index=False)

# 3. Natives Taxonomy

In [199]:
Natives = pd.read_csv(r'../Data Raw/NativeRaw.csv', sep=';')

In [200]:
Natives.drop('AcceptedSpecies', axis=1, inplace=True)

In [204]:
Natives['AcceptedSpecies'] = check_species(Natives['Species'])
Natives['AcceptedSpecies'] = Natives['AcceptedSpecies'].fillna(Natives['Species'])


In [206]:
Natives.to_csv(r'../Data Raw/NativeRaw.csv', sep=';')